# Type 3 Similarity Bands and Subfolders

Partitions the existing Type 3 positive clone set into similarity bands using BCB clone metadata. This does **not** delete or replace `bench_data/bcb_full_type3`; it writes analysis subfolders under `bench_data/bcb_full_type3/similarity_bands/`.

Bands:
- Very Strong: `0.90 <= min(similarity_line, similarity_token) < 1.00`
- Strong: `0.70 <= min(similarity_line, similarity_token) < 0.90`
- Moderate: `0.50 <= min(similarity_line, similarity_token) < 0.70`

In [1]:
import json
import pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parents[2]
elif PROJECT_ROOT.name == "bcb_full_type3":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BCB_DUMP = Path(r"C:\Users\koush\PyProjects\bcb")
BENCH_DIR = PROJECT_ROOT / "bench_data" / "bcb_full_type3"
OUTPUT_ROOT = PROJECT_ROOT.parent / "outputs" / "type3"
SPECTRAL_MANIFEST = OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
GRAPH_MANIFEST = OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
ANALYSIS_OUT = PROJECT_ROOT / "notebooks" / "bcb_full_type3" / "analysis_outputs"
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]

def normalize_pair(left, right):
    left = int(left)
    right = int(right)
    return (left, right) if left <= right else (right, left)

def load_pairs(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            left, right, label = line.strip().split("\t")
            rows.append((int(left), int(right), int(label)))
    return pd.DataFrame(rows, columns=["left_id", "right_id", "label"])

def iter_copy_table(dump_path, table_name, desc):
    dump_path = Path(dump_path)
    target_prefix = f"COPY {table_name} "
    in_table = False
    with dump_path.open("rb") as f, tqdm(total=dump_path.stat().st_size, unit="B", unit_scale=True, desc=desc) as bar:
        for raw in f:
            bar.update(len(raw))
            line = raw.decode("utf-8", errors="replace").rstrip("\n")
            if not in_table:
                if line.startswith(target_prefix):
                    in_table = True
                continue
            if line == r"\.":
                break
            yield line

def parse_float(raw):
    return None if raw == r"\N" or raw == "" else float(raw)

def load_clone_metadata_for_pairs(pair_keys, cache_path=None):
    pair_keys = set(pair_keys)
    if cache_path and Path(cache_path).exists():
        cached = pd.read_csv(cache_path)
        cached["pair_key"] = list(zip(cached.left_id.astype(int), cached.right_id.astype(int)))
        if pair_keys.issubset(set(cached["pair_key"])):
            return cached.drop(columns=["pair_key"])

    rows = []
    remaining = set(pair_keys)
    for line in iter_copy_table(BCB_DUMP, "public.clones", "Scanning BCB clone metadata"):
        parts = line.split("\t")
        if len(parts) < 7:
            continue
        key = normalize_pair(parts[0], parts[1])
        if key not in remaining:
            continue
        sim_line = parse_float(parts[5])
        sim_token = parse_float(parts[6])
        rows.append({
            "left_id": key[0],
            "right_id": key[1],
            "functionality_id": int(parts[2]),
            "bcb_type": parts[3],
            "syntactic_type": int(parts[4]),
            "similarity_line": sim_line,
            "similarity_token": sim_token,
            "min_similarity": min(sim_line, sim_token),
        })
        remaining.remove(key)
        if not remaining:
            break

    meta = pd.DataFrame(rows)
    if cache_path:
        meta.to_csv(cache_path, index=False)
    if remaining:
        print(f"Warning: {len(remaining)} requested pairs were not found in public.clones.")
    return meta

def load_spectral_features_for_ids(method_ids):
    method_ids = {str(int(mid)) for mid in method_ids}
    manifest = json.loads(Path(SPECTRAL_MANIFEST).read_text(encoding="utf-8"))
    features = {}
    for shard_path in tqdm(manifest["shards"], desc="Loading spectral shards", unit="shard"):
        with open(shard_path, "rb") as f:
            shard = pickle.load(f)
        missing = method_ids - set(features)
        for method_id in list(missing):
            if method_id in shard:
                features[method_id] = shard[method_id]
        if len(features) == len(method_ids):
            break
    return features

def informative_eigenvalues(record, graph_type, eps=1e-10):
    if not record:
        return None
    values = np.asarray(record.get(graph_type, {}).get("eigenvalues", []), dtype=float)
    if values.size == 0 or not np.all(np.isfinite(values)) or not np.any(np.abs(values) > eps):
        return None
    return values

def pss(ev1, ev2):
    ev1 = np.asarray(ev1, dtype=float)
    ev2 = np.asarray(ev2, dtype=float)
    nz1 = np.nonzero(ev1)[0]
    nz2 = np.nonzero(ev2)[0]
    if len(nz1) == 0 or len(nz2) == 0:
        return np.nan
    ev1 = ev1[:nz1[-1] + 1]
    ev2 = ev2[:nz2[-1] + 1]
    max_len = max(len(ev1), len(ev2))
    if len(ev1) != max_len:
        ev1 = np.interp(np.linspace(0, 1, max_len), np.linspace(0, 1, len(ev1)), ev1)
    if len(ev2) != max_len:
        ev2 = np.interp(np.linspace(0, 1, max_len), np.linspace(0, 1, len(ev2)), ev2)
    n1 = np.linalg.norm(ev1)
    n2 = np.linalg.norm(ev2)
    if n1 == 0 or n2 == 0:
        return np.nan
    distance = np.linalg.norm(ev1 / n1 - ev2 / n2)
    return float(np.clip((np.sqrt(2) - distance) / np.sqrt(2), 0.0, 1.0))

def node_count(record, graph_type):
    if not record:
        return np.nan
    layer = record.get(graph_type, {})
    return layer.get("nodes", np.nan)

In [2]:
LIMIT_PER_BAND = None  # set to 100 to mirror the SQL LIMIT 100 examples
BANDS = {
    "type3_very_strong": (0.90, 1.00),
    "type3_strong": (0.70, 0.90),
    "type3_moderate": (0.50, 0.70),
}
BANDS_DIR = BENCH_DIR / "similarity_bands"
BANDS_DIR.mkdir(parents=True, exist_ok=True)
print(BANDS_DIR)

c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type3\similarity_bands


In [3]:
train_pos = load_pairs(BENCH_DIR / "train_positives.txt")
train_pos["pair_key"] = [normalize_pair(l, r) for l, r in zip(train_pos.left_id, train_pos.right_id)]
metadata_cache = ANALYSIS_OUT / "type3_positive_clone_metadata.csv"
clone_meta = load_clone_metadata_for_pairs(train_pos["pair_key"], cache_path=metadata_cache)
clone_meta["band"] = None
for band_name, (low, high) in BANDS.items():
    mask = (clone_meta["min_similarity"] >= low) & (clone_meta["min_similarity"] < high)
    clone_meta.loc[mask, "band"] = band_name

display(clone_meta["band"].value_counts(dropna=False).rename_axis("band").reset_index(name="pairs"))

Scanning BCB clone metadata:   0%|          | 0.00/14.0G [00:00<?, ?B/s]

,band,pairs
0,type3_moderate,24490
1,type3_strong,4823
2,type3_very_strong,687


In [4]:
# Load parent code map only once; subfolders will contain only functions referenced by that band.
code_map = {}
with (BENCH_DIR / "data.jsonl").open("r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Loading parent Type 3 code map"):
        obj = json.loads(line)
        code_map[int(obj["idx"])] = obj["func"]
print(f"Loaded code records: {len(code_map):,}")

Loading parent Type 3 code map: 0it [00:00, ?it/s]

Loaded code records: 146,420


In [5]:
created = []
for band_name, (low, high) in BANDS.items():
    band_df = clone_meta[clone_meta["band"] == band_name].copy()
    if LIMIT_PER_BAND is not None:
        band_df = band_df.sort_values(["min_similarity", "left_id", "right_id"], ascending=[False, True, True]).head(LIMIT_PER_BAND)

    out_dir = BANDS_DIR / band_name
    out_dir.mkdir(parents=True, exist_ok=True)

    pair_rows = band_df[["left_id", "right_id"]].drop_duplicates().sort_values(["left_id", "right_id"])
    pair_rows.assign(label=1).to_csv(out_dir / "train_positives.txt", sep="\t", index=False, header=False)
    pair_rows.assign(label=1).to_csv(out_dir / "train.txt", sep="\t", index=False, header=False)
    pair_rows.assign(clone_type="type_3").to_csv(out_dir / "type_labels.tsv", sep="\t", index=False, header=False)
    band_df.to_csv(out_dir / "clone_metadata.csv", index=False)

    needed_ids = set(pair_rows.left_id) | set(pair_rows.right_id)
    with (out_dir / "data.jsonl").open("w", encoding="utf-8") as out:
        for method_id in sorted(needed_ids):
            if method_id not in code_map:
                continue
            out.write(json.dumps({"idx": str(method_id), "func": code_map[method_id]}, ensure_ascii=False))
            out.write("\n")

    metadata = {
        "source_dataset": str(BENCH_DIR),
        "band_name": band_name,
        "min_similarity_lower_inclusive": low,
        "min_similarity_upper_exclusive": high,
        "limit_per_band": LIMIT_PER_BAND,
        "positive_pairs": int(len(pair_rows)),
        "function_ids": int(len(needed_ids)),
        "semantics": "Positive-only Type 3 analysis subset; original full Type 3 dataset is unchanged.",
    }
    (out_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    created.append(metadata)

created_df = pd.DataFrame(created)
display(created_df)
created_df.to_csv(ANALYSIS_OUT / "type3_similarity_band_subfolders.csv", index=False)

,source_dataset,band_name,min_similarity_lower_inclusive,min_similarity_upper_exclusive,limit_per_band,positive_pairs,function_ids,semantics
0,c:\Users\koush\PyProjects\Spectral-Software\be...,type3_very_strong,0.9,1.0,None,687,683,Positive-only Type 3 analysis subset; original...
1,c:\Users\koush\PyProjects\Spectral-Software\be...,type3_strong,0.7,0.9,None,4823,2936,Positive-only Type 3 analysis subset; original...
2,c:\Users\koush\PyProjects\Spectral-Software\be...,type3_moderate,0.5,0.7,None,24490,6024,Positive-only Type 3 analysis subset; original...


In [6]:
# Optional quick inspection of generated subfolders.
for path in sorted(BANDS_DIR.iterdir()):
    if path.is_dir():
        print(path)
        for child in sorted(path.iterdir()):
            print("  ", child.name, child.stat().st_size)

c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type3\similarity_bands\type3_moderate
   clone_metadata.csv 2322419
   data.jsonl 4584386
   metadata.json 401
   train.txt 488200
   train_positives.txt 488200
   type_labels.tsv 610650
c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type3\similarity_bands\type3_strong
   clone_metadata.csv 473784
   data.jsonl 2730435
   metadata.json 398
   train.txt 92440
   train_positives.txt 92440
   type_labels.tsv 116555
c:\Users\koush\PyProjects\Spectral-Software\bench_data\bcb_full_type3\similarity_bands\type3_very_strong
   clone_metadata.csv 72882
   data.jsonl 817180
   metadata.json 401
   train.txt 13262
   train_positives.txt 13262
   type_labels.tsv 16697
